In [6]:
import sys
!{sys.executable} -m pip install tensorflow

In [7]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [9]:
df = pd.read_csv("Churn_Modelling.csv")

print(df.head())
print(df.shape)
print(df.info())

   RowNumber  CustomerId   Surname  CreditScore  ... HasCrCard IsActiveMember  EstimatedSalary  Exited
0          1    15634602  Hargrave          619  ...         1              1        101348.88       1
1          2    15647311      Hill          608  ...         0              1        112542.58       0
2          3    15619304      Onio          502  ...         1              0        113931.57       1
3          4    15701354      Boni          699  ...         0              0         93826.63       0
4          5    15737888  Mitchell          850  ...         1              1         79084.10       0

[5 rows x 14 columns]
(10000, 14)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore    

In [10]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

print(df.head())
print(df.shape)

   CreditScore Geography  Gender  Age  Tenure  ...  NumOfProducts  HasCrCard  IsActiveMember  EstimatedSalary  Exited
0          619    France  Female   42       2  ...              1          1               1        101348.88       1
1          608     Spain  Female   41       1  ...              1          0               1        112542.58       0
2          502    France  Female   42       8  ...              3          1               0        113931.57       1
3          699    France  Female   39       1  ...              2          0               0         93826.63       0
4          850     Spain  Female   43       2  ...              1          1               1         79084.10       0

[5 rows x 11 columns]
(10000, 11)


In [11]:
X = df.drop('Exited', axis=1)
y = df['Exited']

In [12]:
X = pd.get_dummies(X, columns=['Geography'], dtype=int) # one-hot encoding - convert categorical to numerical form

le = LabelEncoder()
X['Gender'] = le.fit_transform(X['Gender'])

print(X.head())

   CreditScore  Gender  Age  Tenure  ...  EstimatedSalary  Geography_France  Geography_Germany  Geography_Spain
0          619       0   42       2  ...        101348.88                 1                  0                0
1          608       0   41       1  ...        112542.58                 0                  0                1
2          502       0   42       8  ...        113931.57                 1                  0                0
3          699       0   39       1  ...         93826.63                 1                  0                0
4          850       0   43       2  ...         79084.10                 0                  0                1

[5 rows x 12 columns]


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape)
print(X_test.shape)

(8000, 12)
(2000, 12)


In [14]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [15]:
model = Sequential()

model.add(tf.keras.Input(shape=(12,)))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [17]:
model.compile(
    optimizer='adam', # how the model runs
    loss='binary_crossentropy', # how model measures mistakes
    metrics=['accuracy'] # how we evaluate model performance
)

In [19]:
print(X_train.shape)
print(X_test.shape)

(8000, 12)
(2000, 12)


In [20]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6394 - loss: 0.6370 - val_accuracy: 0.8012 - val_loss: 0.4850
Epoch 2/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8016 - loss: 0.4569 - val_accuracy: 0.8275 - val_loss: 0.4290
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8120 - loss: 0.4280 - val_accuracy: 0.8250 - val_loss: 0.4134
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8252 - loss: 0.4125 - val_accuracy: 0.8381 - val_loss: 0.4022
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8344 - loss: 0.3962 - val_accuracy: 0.8400 - val_loss: 0.3907
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8441 - loss: 0.3793 - val_accuracy: 0.8469 - val_loss: 0.3766
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8537 - loss: 0.3654 - val_accuracy: 0.8537 - val_loss: 0.3680
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8572 - loss: 0.3566 - val_accuracy: 0.

In [24]:
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  


In [25]:
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy * 100, "%")

Model Accuracy: 85.6 %


In [26]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1515   92]
 [ 196  197]]
